# Locked Test Evaluation Review

Review the already-generated final held-out test artifacts. This notebook reads saved CSVs only; it does not train models, tune post-processing, or rerun the locked test protocol. Use it as a compact companion to `scripts/run_final_test_evaluation.py` when checking the final results.


In [ ]:
import json
from pathlib import Path

import pandas as pd

run_dir = Path("../outputs/runs/final_test_evaluation")
analysis_dir = run_dir / "test_error_analysis"

## Frozen Protocol

Load the JSON protocol written by the final evaluation script. This records the input artifacts, selected candidate models, and post-processing rules so the locked-test comparison can be audited without re-opening model-selection decisions.


In [ ]:
protocol = json.loads((run_dir / "final_test_protocol.json").read_text())
protocol

## Primary Comparison Table

Display the validation-to-test comparison table for the finalized candidates and variants. The table keeps validation macro F1 next to locked-test macro and per-class F1 so the final results can be compared with earlier validation expectations and with the prior DREAMT project.


In [ ]:
comparison = pd.read_csv(run_dir / "final_comparison_table.csv")
display_columns = [
    "candidate",
    "ablation",
    "base_model",
    "model",
    "variant",
    "validation_macro_f1",
    "test_macro_f1",
    "test_Wake_f1",
    "test_Non_REM_f1",
    "test_REM_f1",
]
comparison[display_columns].sort_values("test_macro_f1", ascending=False)

## Error Summaries

Inspect compact error-analysis tables generated from the locked-test predictions. The cells below review per-class behavior, the weakest participant-level rows, and performance by distance to the nearest true sleep-stage transition.


In [ ]:
per_class = pd.read_csv(analysis_dir / "metrics" / "per_model_per_class_metrics.csv")
per_class.sort_values(["candidate", "model", "label"]).head(30)

In [ ]:
per_participant = pd.read_csv(analysis_dir / "metrics" / "per_participant_metrics.csv")
per_participant.sort_values("macro_f1").head(20)

In [ ]:
transition = pd.read_csv(analysis_dir / "metrics" / "transition_distance_metrics.csv")
transition.sort_values(["candidate", "model", "transition_distance_bin"]).head(40)